# Tests: `fasterai.misc.bn_folding` (source `nbs/misc/bn_folding.ipynb`)

In [ ]:
from fastcore.test import *
import torch
import torch.nn as nn
from fasterai.misc.bn_folding import *

In [ ]:
from fastcore.test import *

# Fold preserves forward pass
model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32)
).eval()
x = torch.randn(2, 3, 8, 8)

with torch.no_grad():
    out_orig = model(x)
    folder = BN_Folder()
    model_f = folder.fold(model)
    out_fold = model_f(x)

test_close(out_orig, out_fold, eps=1e-5)

# BN layers replaced with Identity after folding
bn_count = sum(1 for m in model_f.modules() if isinstance(m, nn.BatchNorm2d))
test_eq(bn_count, 0)

# Folded conv has bias (even if original didn't)
assert model_f[0].bias is not None
assert model_f[3].bias is not None

# Identity modules present where BN was
id_count = sum(1 for m in model_f.modules() if isinstance(m, nn.Identity))
test_eq(id_count, 2)